# E791 $D^+\to\pi^-\pi^+\pi^+$ — GenFit study

Repeated coefficient-fit closure test using the native `GenFit` implementation. The production configuration is **500 pseudoexperiments × 50,000 events**. Resonance masses, widths, spins and meson radii are fixed; the $\rho(770)\pi^+$ coefficient is fixed to $1+0i$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, GenFit, NonResonant, Parameter,
    RealImag, Resonance, enable_x64,
)
enable_x64()


## 1. E791 Fit-2 model


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
fit2_polar = {
    "sigma": (1.17,205.7), "rho770": (1.00,0.0), "NR": (0.48,57.3),
    "f0_980": (0.43,165.0), "f2_1270": (0.76,57.3),
    "f0_1370": (0.26,105.4), "rho1450": (0.14,319.1),
}
def polar_to_xy(r, phase_deg):
    phase=np.deg2rad(phase_deg); return r*np.cos(phase), r*np.sin(phase)
def internal_xy(name):
    r,phase=fit2_polar[name]
    if name=="NR": phase += 180.0
    return polar_to_xy(r,phase)
truth_xy={name:internal_xy(name) for name in fit2_polar}


In [ ]:
truth={}
def free_coefficient(name):
    x,y=truth_xy[name]; truth[f"{name}.x"]=float(x); truth[f"{name}.y"]=float(y)
    return RealImag(Parameter.coefficient(f"{name}.x",0.0,owner=name,step=0.01), Parameter.coefficient(f"{name}.y",0.0,owner=name,step=0.01))
coefficients={
    "sigma":free_coefficient("sigma"), "rho770":RealImag(1.0,0.0),
    "NR":free_coefficient("NR"), "f0_980":free_coefficient("f0_980"),
    "f2_1270":free_coefficient("f2_1270"), "f0_1370":free_coefficient("f0_1370"),
    "rho1450":free_coefficient("rho1450"),
}
components=[
    Resonance("sigma",(0,1),coefficients["sigma"],mass=0.4780,width=0.3240,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho770",(0,1),coefficients["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_980",(0,1),coefficients["f0_980"],mass=0.9750,width=0.0440,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f2_1270",(0,1),coefficients["f2_1270"],mass=1.2750,width=0.1850,spin=2,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_1370",(0,1),coefficients["f0_1370"],mass=1.4340,width=0.1730,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho1450",(0,1),coefficients["rho1450"],mass=1.4650,width=0.3100,spin=1,resonance_radius=3.0,parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]
model=DecayModel(channel,components)
print(f"free parameters: {len([p for p in model.parameters if not p.fixed])}")


## 2. Configure the production GenFit

The candidate pool, component amplitudes and fixed normalization matrix are cached and reused across all pseudoexperiments.


In [ ]:
N_FITS=500
SAMPLE_SIZE=50_000
study=GenFit(
    model, n_fits=N_FITS, sample_size=SAMPLE_SIZE, truth_values=truth,
    grid_resolution=1000, pool_size=1_000_000, start_range=(-2.5,2.5),
    seed=791, pool_seed=2000, ncall=100_000, tolerance=1e-4, verbose=1,
)
study


## 3. Run 500 pseudoexperiments


In [ ]:
result=study.run()


## 4. Statistical summary and convergence


In [ ]:
result.print_summary()
valid=result.valid_mask
print(f"valid fits   : {result.n_valid}/{result.n_fits}")
print(f"success rate : {100*result.success_rate:.2f}%")
print(f"mean EDM     : {np.mean(result.edm[valid]):.4e}")
print(f"median nfcn  : {np.median(result.nfcn[valid]):.0f}")
print(f"mean fit-truth NLL: {np.mean(result.nll[valid]-result.truth_nll[valid]):.6f}")


## 5. Fitted-parameter histograms with Gaussian fits


In [ ]:
for name in result.parameter_names:
    fig,ax=plt.subplots(figsize=(7,4.8))
    result.plot(name,bins=30,ax=ax)
    g=result.gaussian_fit(name)
    ax.text(0.03,0.97,f"mean = {g.mean:.5f} ± {g.mean_error:.5f}\nsigma = {g.sigma:.5f} ± {g.sigma_error:.5f}",transform=ax.transAxes,va="top")
    fig.tight_layout(); plt.show()


## 6. NLL distribution


In [ ]:
fig,ax=plt.subplots(figsize=(7,4.8))
result.plot("nll",bins=30,ax=ax)
g=result.gaussian_fit("nll")
ax.text(0.03,0.97,f"mean = {g.mean:.3f} ± {g.mean_error:.3f}\nsigma = {g.sigma:.3f} ± {g.sigma_error:.3f}",transform=ax.transAxes,va="top")
fig.tight_layout(); plt.show()


## 7. Pull distributions

For an unbiased, correctly calibrated fitter the pulls should be approximately Gaussian with mean 0 and width 1.


In [ ]:
for name in result.parameter_names:
    pulls=result.pulls(name)
    fig,ax=plt.subplots(figsize=(7,4.8))
    ax.hist(pulls,bins=30,alpha=0.65)
    ax.axvline(0.0,linestyle="--",label="expected mean")
    ax.set(xlabel=f"pull({name})",ylabel="Pseudoexperiments",title=f"GenFit pull: {name}")
    ax.legend(); fig.tight_layout(); plt.show()


## 8. Compact closure table


In [ ]:
print(f"{'parameter':18s} {'truth':>12s} {'gauss mean':>14s} {'mean-truth':>14s} {'gauss sigma':>14s}")
for name in result.parameter_names:
    g=result.gaussian_fit(name); t=truth[name]
    print(f"{name:18s} {t:12.6f} {g.mean:14.6f} {g.mean-t:14.6f} {g.sigma:14.6f}")
